# AgroPest-12 Part B — 02b: BYOL Backbone → YOLOv12m Fine-tune @ 20% Labels

**Pair:** `partB-02a-byol-pretrain` → `partB-02b-byol-detect`
**Detector:** YOLOv12m (`yolo12m.yaml`) — Part A winner (test mAP50–95 = 0.4773)
**Label budget:** ρ = 0.20 → 2,295 labelled images (`finetune_20` partition)

This notebook runs **one arm only**: the BYOL-initialised downstream fine-tune, followed by
validation/test evaluation, per-class metrics, and a 36-image per-class detection panel with a
correctness log.

The **random-init and COCO-pretrained baselines** are trained in a separate notebook
(`partb-00-baselines`) under the identical frozen protocol, so all Part B arms stay comparable.

## Pipeline

```
AgroPest-12-PartB-Safe (Kaggle input):
  finetune_20/   2,295 labelled images   <- fine-tune only
  valid/         1,089 labelled images   <- model selection only
  test/            543 labelled images   <- final evaluation only

partb-02a output (Kaggle input):
  byol_best_backbone_yolo12m.pt        <- SSL backbone only (402 tensors)

02b flow:
  1. verify D^0.2 and disjointness (finetune vs valid vs test)
  2. build a fresh random yolo12m.yaml detector
  3. weight surgery: copy the 402 SSL backbone tensors into modules 0..8
     (neck + head stay random) -> mandatory mapping report printed
  4. fine-tune 50 epochs on finetune_20
  5. evaluate val + test (P/R/mAP50/mAP50-95), PR curves, per-class table
  6. detection panel: 3 test images per class + per-class correctness log
  7. export the BYOL-arm detector and run manifest
```

In [ ]:
# Kaggle already ships a CUDA-matched PyTorch build. Do not install a different torch wheel.
%pip install -q --upgrade "ultralytics>=8.4.96" "scikit-learn>=1.3" seaborn

## Runtime Environment

Run on **Kaggle GPU T4 x2**. At ρ = 0.2 the fine-tune set is only 2,295 images, so the single
50-epoch arm plus the evaluations fit inside a 9-hour T4 session.

In [ ]:
import json
import os
import random
import shutil
import sys
import time
import warnings
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml

import ultralytics
from ultralytics import YOLO

warnings.filterwarnings("ignore")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print("Python    :", sys.version.split()[0])
print("PyTorch   :", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA      :", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "Select Kaggle GPU T4 x2."
for gpu in range(torch.cuda.device_count()):
    print(f"  GPU {gpu}: {torch.cuda.get_device_name(gpu)}")


## 1. Configuration

Every parameter below is commented for the viva. The frozen protocol is shared with the separate
baselines notebook, so the BYOL arm stays comparable with random-init and COCO-init runs.
`fast_run` is a documented 1-epoch smoke test of the whole pipeline; set it to `False` for the real
50-epoch run.

In [ ]:
@dataclass
class CFG:
    # ------------- 1. shared Part B data & reproducibility -------------
    partb_root: str = (
        "/kaggle/input/datasets/sadikshahriar/"
        "agropest-12-partb-safe/AgroPest-12-PartB-Safe"
    )  # Published leakage-safe dataset; every Part B notebook reads it.
    seed: int = 445                # Part B shared seed (sampling, DDP, augmentation).
    label_fraction: float = 0.20   # rho = 20% label budget -> finetune_20.

    # ------------- 2. SSL checkpoint (published output of Notebook 02a) -------------
    checkpoint_root: str = (
        "/kaggle/input/notebooks/sadikshahriar/"
        "partb-02a-byol-pretrain/partB_02a_byol_yolo12m"
    )  # Attach the notebook output as a Kaggle "Input".
    backbone_checkpoint: str = "byol_best_backbone_yolo12m.pt"  # backbone-only export from 02a.
    output_dir: str = "/kaggle/working/partB_02b_byol_yolo12m_detect"

    # ------------- 3. architecture -------------
    model_yaml: str = "yolo12m.yaml"  # Part A winner; YAML = random weights, SSL backbone loaded via surgery.

    # ------------- 4. FIXED fine-tuning protocol (shared with baselines notebook) -------------
    epochs: int = 50             # Part B official budget for every downstream run.
    image_size: int = 640        # native dataset resolution (Part A).
    batch: int = 32              # 16 per GPU on T4 x2 (DDP).
    workers: int = 4             # per-GPU dataloader workers (8 across both DDP ranks).
    amp: bool = True             # FP16 halves VRAM on T4; matches Part A.
    optimizer: str = "auto"      # iterations = ceil(2295/64) * 50 ~= 1,800 < 10k -> resolves to AdamW (not MuSGD). 'auto' is kept so every arm + the baselines notebook select identically.
    lr0: float = 0.01            # peak LR; only used if optimizer is explicit (auto overrides it with the AdamW fit LR).
    lrf: float = 0.01            # final LR = lr0*lrf = 1e-4.
    momentum: float = 0.937      # momentum for SGD-family optimizers (AdamW under 'auto' uses betas=(0.9, 0.999)).
    weight_decay: float = 5e-4   # L2 regularisation (scaled by nbs/batch internally, stays 5e-4).
    warmup_epochs: float = 3.0   # stabilises early gradients for any init.
    warmup_momentum: float = 0.8
    warmup_bias_lr: float = 0.1
    close_mosaic: int = 10       # refine boxes on real crops in the last 10 epochs (Part A).
    mosaic: float = 1.0          # helps rare classes (12-class imbalance).
    mixup: float = 0.15          # mild to avoid confusing small pest features.
    degrees: float = 10.0        # arbitrary field orientation.
    scale: float = 0.5           # 0.75x-1.25x size variation.
    shear: float = 0.0           # no shear (distorts insect morphology).
    translate: float = 0.1       # partially visible edge pests.
    fliplr: float = 0.5
    flipud: float = 0.5          # top-down camera: any angle.
    hsv_h: float = 0.01          # keep hue; colour is a key species cue.
    hsv_s: float = 0.4           # reduced vs default 0.7 to preserve colour classes.
    hsv_v: float = 0.4           # outdoor lighting variation.
    cos_lr: bool = False         # linear LR decay (Part A protocol).
    nbs: int = 64                # nominal batch for accumulate energy: round(nbs/batch)=2 -> BN sees 64-image batches; also drives LR-schedule iterations & weight-decay scaling.
    patience: int = 0            # no early stopping -> exactly 50 epochs for every arm.
    deterministic: bool = True   # reproducible across runs.

    # ------------- 5. inference -------------
    conf: float = 0.25           # detection confidence threshold (Part A).
    iou: float = 0.70            # NMS IoU (Part A).
    max_det: int = 300           # cap detections per image.

    # ------------- 6. reference (informational only) -------------
    parta_test_map50_95: float = 0.4773  # full-label Part A YOLOv12m; never used in decisions.

    fast_run: bool = False       # True -> 1-epoch smoke test of the pipeline.


cfg = CFG()
seed_everything(cfg.seed)
sns.set_theme(style="whitegrid")

PARTB_ROOT = Path(cfg.partb_root)
OUTPUT_DIR = Path(cfg.output_dir)
RUNS_DIR = OUTPUT_DIR / "runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "Ants", "Bees", "Beetles", "Caterpillars", "Earthworms", "Earwigs",
    "Grasshoppers", "Moths", "Slugs", "Snails", "Wasps", "Weevils",
]

EPOCHS = 1 if cfg.fast_run else cfg.epochs
IMAGE_SIZE = cfg.image_size
BATCH = cfg.batch
DEVICE = [0, 1]  # DDP across both T4 GPUs

(OUTPUT_DIR / "partb_02b_config.json").write_text(json.dumps(asdict(cfg), indent=2), encoding="utf-8")
print(json.dumps(asdict(cfg), indent=2))
print("active epochs:", EPOCHS, "| FAST RUN:", cfg.fast_run)


## 2. Data — Subset D^0.2, Disjointness, data.yaml

The published dataset already contains the 20% labelled subset. The *subset builder* therefore
*verifies* the partition: counts, image/label pairing, filename-intersection checks, then writes a
working `data.yaml` with absolute paths to `/kaggle/working/` only.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_images(folder: Path) -> list:
    return sorted(p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)


def list_labels(folder: Path) -> list:
    return sorted(p for p in folder.rglob("*.txt") if p.is_file())


def classes_in_label(label_file: Path) -> Counter:
    counts = Counter()
    for line in label_file.read_text(encoding="utf-8").splitlines():
        fields = line.split()
        if len(fields) >= 5:
            counts[int(float(fields[0]))] += 1
    return counts


assert PARTB_ROOT.is_dir(), f"Attach AgroPest-12-PartB-Safe. Missing: {PARTB_ROOT}"

FINETUNE_ROOT = PARTB_ROOT / "finetune_20"
VALID_ROOT = PARTB_ROOT / "valid"
TEST_ROOT = PARTB_ROOT / "test"

fin_images = list_images(FINETUNE_ROOT / "images")   # 2,295
val_images = list_images(VALID_ROOT / "images")      # 1,089
test_images = list_images(TEST_ROOT / "images")      #   543
fin_labels = list_labels(FINETUNE_ROOT / "labels")

EXPECTED = {"finetune_20": 2295, "valid": 1089, "test": 543}
assert len(fin_images) == EXPECTED["finetune_20"], len(fin_images)
assert len(val_images) == EXPECTED["valid"], len(val_images)
assert len(test_images) == EXPECTED["test"], len(test_images)
assert len(fin_labels) == len(fin_images), "finetune_20 image/label pairing must be 1:1"

fin_names, val_names, test_names = ({p.stem for p in s} for s in (fin_images, val_images, test_images))
assert not (fin_names & val_names), "finetune vs valid overlap"
assert not (fin_names & test_names), "finetune vs test overlap"
assert not (val_names & test_names), "valid vs test overlap"

print("partition image/label counts:")
for tag, imgs, lbls in (
    ("finetune_20", fin_images, fin_labels),
    ("valid", val_images, list_labels(VALID_ROOT / "labels")),
    ("test", test_images, list_labels(TEST_ROOT / "labels")),
):
    print(f"  {tag:>12}: {len(imgs):>5} / {len(lbls):>5}")
print("disjointness:", {
    "finetune_vs_valid": not (fin_names & val_names),
    "finetune_vs_test": not (fin_names & test_names),
    "valid_vs_test": not (val_names & test_names),
})

# Resolve the BYOL backbone checkpoint robustly across Kaggle mounts.
CKPT = None
candidate = Path(cfg.checkpoint_root) / cfg.backbone_checkpoint
if candidate.is_file():
    CKPT = candidate
else:
    hits = list(Path("/kaggle/input").rglob(cfg.backbone_checkpoint))
    CKPT = hits[0] if hits else None
assert CKPT is not None and CKPT.is_file(), f"Checkpoint not found: {cfg.backbone_checkpoint}"
print("SSL backbone checkpoint:", CKPT)

# data.yaml with absolute paths -> written to /kaggle/working, never /kaggle/input.
DATA_YAML = OUTPUT_DIR / "data_20.yaml"
data_config = {
    "path": str(PARTB_ROOT),
    "train": "finetune_20/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding="utf-8")
print("wrote", DATA_YAML)


## 3. D^0.2 Instance Counts and Class Balance

Read the `finetune_20` label files directly to report per-class instance counts and to cross-check
the published `class_distribution_report.csv` that every Part B notebook must cite.

In [ ]:
fin_stats = Counter()
for lbl in fin_labels:
    fin_stats.update(classes_in_label(lbl))

inst_df = pd.DataFrame({"class": CLASS_NAMES,
                        "finetune_instances": [fin_stats[c] for c in range(12)]})
inst_df["pct"] = 100 * inst_df["finetune_instances"] / inst_df["finetune_instances"].sum()

report_csv = PARTB_ROOT / "class_distribution_report.csv"
if report_csv.is_file():
    pub = pd.read_csv(report_csv)
    inst_df = inst_df.merge(pub[["class", "finetune_pct"]].rename(
        columns={"finetune_pct": "published_pct"}), on="class", how="left")
display(inst_df.round(2))
print("TOTAL instances in D^0.2:", int(inst_df["finetune_instances"].sum()))

ax = inst_df.plot.bar(x="class", y="finetune_instances", figsize=(10, 3.4), color="#1261A0", legend=False)
ax.set_title("label instances per pest class - finetune_20 subset")
ax.set_ylabel("instances")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "finetune_class_stats.png", dpi=180)
plt.show()


## 4. Weight Surgery — SSL Backbone into a Fresh YOLOv12m

`byol_best_backbone_yolo12m.pt` stores **backbone-only** tensors keyed as the Notebook 02a encoder
(`layers.<n>.<module>` = YOLOv12m backbone modules 0..8). The BYOL projector and predictor were discarded in 02a (encoder transferred only).

Surgery does 4 things:
1. builds a **fresh** random `yolo12m.yaml` detector (`ssl_detector`);
2. copies each SSL tensor to the matching `model.<n>.<module>` key;
3. **enforces 100% backbone coverage** — a silent partial load is a hard error;
4. leaves neck + head randomly initialised and prints the module-by-module mapping.

It then **exports the surgically modified detector to `ssl_surged_yolo12m.pt`**.
This file hand-off is mandatory: ultralytics rebuilds the training model from the `model`
argument string and, under DDP (`device=[0,1]`), re-spawns a subprocess from serialised args,
so an *in-memory* weight swap would be silently discarded. Fine-tuning therefore starts from the
exported checkpoint.

The mapping table below is the mandatory lab deliverable (which layers came from SSL, which random).

In [ ]:
checkpoint = torch.load(str(CKPT), map_location="cpu", weights_only=False)
assert checkpoint.get("format_version") == 1, checkpoint.keys()
assert checkpoint.get("yolo_model") == cfg.model_yaml, checkpoint.get("yolo_model")
bb_state = checkpoint["encoder_state_dict"]

ssl_detector = YOLO(cfg.model_yaml)            # fresh random model = the "blank" detector
det = ssl_detector.model
target = det.state_dict()
backbone_end = len(det.yaml["backbone"])       # modules 0 .. backbone_end-1

matched: dict = {}
skipped_keys: list = []
for src_key, tensor in bb_state.items():
    parts = src_key.split(".")
    while parts and not parts[0].isdigit():    # strip the "layers." prefix
        parts = parts[1:]
    if not parts:
        continue
    mapped = "model." + ".".join(parts)
    if mapped in target and target[mapped].shape == tensor.shape:
        matched[mapped] = tensor
    else:
        skipped_keys.append(src_key)


def module_idx(key: str) -> int:
    return int(key.split(".")[1])


backbone_keys = {k for k in target if module_idx(k) < backbone_end}
matched_backbone = {k for k in matched if module_idx(k) < backbone_end}

# Guarantee no silent partial weight load.
assert matched_backbone == backbone_keys, "backbone coverage must be 100%"
# No matched key may live outside the backbone (projector / neck / head must stay random).
assert all(module_idx(k) < backbone_end for k in matched), "mapped keys must belong to the backbone"

det.load_state_dict(matched, strict=False)  # strict=False is intentional: unmatched stay random.

# CRITICAL: ultralytics builds the training model from the `model` argument string and, under
# DDP (device=[0,1]), re-spawns a fresh subprocess from serialised args -- an in-memory weight
# swap here would be silently discarded and training would start from random weights. Export the
# surgically modified detector as a checkpoint and launch fine-tuning from that file instead.
import copy

SURGED_PT = OUTPUT_DIR / "ssl_surged_yolo12m.pt"
ssl_detector.model.args = {"model": cfg.model_yaml, "task": "detect"}
torch.save(
    {
        "model": copy.deepcopy(ssl_detector.model).half(),
        "ema": None,
        "optimizer": None,
        "epoch": 0,
        "train_args": dict(ssl_detector.model.args),
    },
    SURGED_PT,
)
assert SURGED_PT.is_file(), "surgical export missing"
print("Surgical export (fine-tune source):", SURGED_PT)

rows = []
for idx in range(len(det.model)):
    layer_keys = {k for k in target if module_idx(k) == idx}
    covered = sum(1 for k in matched if module_idx(k) == idx)
    rows.append({
        "module_idx": idx,
        "type": "SSL-backbone" if idx < backbone_end else "random (neck/head)",
        "module": str(type(det.model[idx]).__name__).split(".")[-1],
        "total_keys": len(layer_keys),
        "ssl_matched": covered,
        "coverage": covered / max(1, len(layer_keys)),
    })
surgery_df = pd.DataFrame(rows)
display(surgery_df)

surgery_report = {
    "source": str(CKPT),
    "target_model": cfg.model_yaml,
    "surgical_export": str(SURGED_PT),
    "backbone_end": backbone_end,
    "matched_keys": len(matched),
    "skipped_keys": len(skipped_keys),
    "backbone_keys": len(backbone_keys),
    "backbone_coverage": len(matched_backbone) / len(backbone_keys),
    "projector_transferred": False,
    "detector_neck_head_transferred": False,
}
(OUTPUT_DIR / "weight_surgery_report.json").write_text(json.dumps(surgery_report, indent=2), encoding="utf-8")
print(json.dumps(surgery_report, indent=2))
assert surgery_report["backbone_coverage"] == 1.0


## 5. Fine-tuning — BYOL-initialised Detector (50 epochs)

`run_fine_tune` loads the **exported** surgical checkpoint (`ssl_surged_yolo12m.pt`) and applies the
frozen Part B protocol. The baseline arms in the separate notebook use the identical
hyper-parameters. After training, a single validation image is rendered for a quick
eye-check — before the full metric evaluation.

In [ ]:
def run_fine_tune(model_file: Path, project: Path, name: str) -> Path:
    # Train from the exported checkpoint (not the in-memory object): the trainer rebuilds the
    # model from the `model` arg string and DDP spawns a fresh subprocess, so only a file
    # survives the hand-off. YOLO(str(.pt)) -> ckpt set -> Model.train injects these weights.
    model = YOLO(str(model_file))
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        workers=cfg.workers,
        device=DEVICE,
        amp=cfg.amp,
        optimizer=cfg.optimizer,
        lr0=cfg.lr0,
        lrf=cfg.lrf,
        momentum=cfg.momentum,
        weight_decay=cfg.weight_decay,
        warmup_epochs=cfg.warmup_epochs,
        warmup_momentum=cfg.warmup_momentum,
        warmup_bias_lr=cfg.warmup_bias_lr,
        close_mosaic=cfg.close_mosaic,
        mosaic=cfg.mosaic,
        mixup=cfg.mixup,
        degrees=cfg.degrees,
        scale=cfg.scale,
        shear=cfg.shear,
        translate=cfg.translate,
        fliplr=cfg.fliplr,
        flipud=cfg.flipud,
        hsv_h=cfg.hsv_h,
        hsv_s=cfg.hsv_s,
        hsv_v=cfg.hsv_v,
        cos_lr=cfg.cos_lr,
        nbs=cfg.nbs,
        patience=cfg.patience,
        deterministic=cfg.deterministic,
        seed=cfg.seed,
        project=str(project),
        name=name,
        exist_ok=True,
        plots=True,
        verbose=True,
    )
    return project / name / "weights" / "best.pt"


start = time.perf_counter()
SSL_BEST = run_fine_tune(SURGED_PT, RUNS_DIR / "byol", "yolo12m_byol_20pct")
print(f"BYOL fine-tune finished in {(time.perf_counter() - start) / 60:.1f} min")
assert Path(SSL_BEST).is_file(), "no best.pt after fine-tuning"
print("best:", SSL_BEST)

# quick demo inference on a single validation image BEFORE full evaluation
demo_img = val_images[0]
demo_result = YOLO(str(SSL_BEST)).predict(
    source=str(demo_img), imgsz=IMAGE_SIZE, conf=cfg.conf, iou=cfg.iou,
    device=0 if torch.cuda.device_count() >= 1 else "cpu", verbose=False)[0]
plt.figure(figsize=(6.5, 6.5))
plt.imshow(cv2.cvtColor(demo_result.plot(), cv2.COLOR_BGR2RGB))
plt.title("BYOL-initialised detector on a validation image\n" + demo_img.name, fontsize=9)
plt.axis("off")
plt.tight_layout()
plt.show()


## 6. Training Curves

`results.csv` written by Ultralytics during training. In a full run (50 epochs) this shows the
SSL-initialised detector converging on the 20% label budget.

In [ ]:
hist_csv = RUNS_DIR / "byol" / "yolo12m_byol_20pct" / "results.csv"
if hist_csv.is_file():
    hist = pd.read_csv(hist_csv)
    hist.columns = [c.strip() for c in hist.columns]
    cols = [c for c in ("metrics/mAP50(B)", "metrics/mAP50-95(B)") if c in hist.columns]
    if cols:
        ax = hist.plot(x="epoch", y=cols, marker="o", figsize=(10, 4.2),
                       title=f"BYOL-init detector, {EPOCHS} epochs - training curves",
                       color=["#1261A0", "#167C3A"])
        ax.set_ylim(0, 1)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("results.csv missing:", hist_csv)


## 7. Evaluation — Validation and Test

Score the fine-tuned BYOL detector on both held-out splits with the Ultralytics val mode (same
protocol as Part A) and save the summary table.

In [ ]:
def evaluate_arm(model: YOLO, split: str, arm_name: str) -> dict:
    metrics = model.val(
        data=str(DATA_YAML),
        split=split,
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        device=0 if torch.cuda.device_count() >= 1 else "cpu",
        plots=True,
        project=str(OUTPUT_DIR / "evaluation"),
        name=arm_name + "_" + split,
        exist_ok=True,
    )
    return {
        "arm": arm_name,
        "split": split,
        "precision": float(metrics.box.p.mean()),
        "recall": float(metrics.box.r.mean()),
        "map50": float(metrics.box.map50.mean()),
        "map50_95": float(metrics.box.map.mean()),
        "map75": float(metrics.box.map75.mean()),
        "f1": float(metrics.box.f1.mean()),
    }


summary = pd.DataFrame([
    evaluate_arm(YOLO(str(SSL_BEST)), "val", "byol"),
    evaluate_arm(YOLO(str(SSL_BEST)), "test", "byol"),
])
summary.to_csv(OUTPUT_DIR / "detection_performance.csv", index=False)
display(summary.round(4))


## 8. Precision–Recall Curve — BYOL detector (test set)

The val run above saved its diagnostic plots. The PR curve shows the operating range of the
SSL-initialised detector at ρ = 0.2.

In [ ]:
from matplotlib.image import imread


def show_eval_plot(subdir: str, filename: str) -> None:
    path = OUTPUT_DIR / "evaluation" / subdir / filename
    if not path.is_file():
        hits = list(OUTPUT_DIR.rglob(filename))
        path = hits[0] if hits else None
    if path is not None:
        plt.figure(figsize=(7, 5))
        plt.imshow(imread(path))
        plt.axis("off")
        plt.tight_layout()
        plt.show()


show_eval_plot("byol_test", "BoxPR_curve.png")
show_eval_plot("byol_test", "BoxP_curve.png")


## 9. Per-Class Test Metrics

Per-class AP50 and AP50–95 on the test split show *where* the 20% label budget hurts: rare or
visually confusable pests (e.g. Slugs ↔ Snails).

In [ ]:
def per_class_test(best_path: Path) -> pd.DataFrame:
    metrics = YOLO(str(best_path)).val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        device=0 if torch.cuda.device_count() >= 1 else "cpu",
        plots=False,
        project=str(OUTPUT_DIR / "evaluation"),
        name="byol_test_perclass",
        exist_ok=True,
    )
    box = metrics.box
    return pd.DataFrame({
        "class": [CLASS_NAMES[int(i)] for i in box.ap_class_index],
        "precision": np.asarray(box.p),
        "recall": np.asarray(box.r),
        "f1": np.asarray(box.f1),
        "ap50": np.asarray(box.ap50),
        "ap50_95": np.asarray(box.ap),
    }).set_index("class")


per_class_df = per_class_test(SSL_BEST)
display(per_class_df.round(4))
per_class_df.to_csv(OUTPUT_DIR / "test_per_class_metrics.csv")


## 10. Detection Panel — 3 Test Images per Class + Correctness Log

Following the Part A protocol, 3 test images are selected per pest class (36 images total), each
containing at least one instance of its target class. The grid shows the BYOL detector's boxes
(colour-coded per class), and the correctness log below scores each image: `OK` if every GT box of
the class is matched by a same-class prediction at IoU ≥ 0.5, `MISS` otherwise.

In [ ]:
model = YOLO(str(SSL_BEST))

# ------------- 1. find 3 test images per class by scanning label files -------------
test_img_dir = TEST_ROOT / "images"
test_lbl_dir = TEST_ROOT / "labels"

CLASS_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8",
    "#f58231", "#911eb4", "#46f0f0", "#f032e6",
    "#bcf60c", "#fabebe", "#008080", "#e6beff",
]

class_images = {i: [] for i in range(len(CLASS_NAMES))}
for fname in sorted(os.listdir(test_lbl_dir)):
    if not fname.endswith(".txt"):
        continue
    img_stem = os.path.splitext(fname)[0]
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = test_img_dir / (img_stem + ext)
        if candidate.exists():
            img_path = candidate
            break
    if img_path is None:
        continue
    lbl_path = test_lbl_dir / fname
    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cid = int(float(parts[0]))
                if cid in class_images and len(class_images[cid]) < 3:
                    class_images[cid].append((img_path, lbl_path))
    if all(len(v) >= 3 for v in class_images.values()):
        break

print("Images selected per class:")
for cid in range(len(CLASS_NAMES)):
    print(f"  {CLASS_NAMES[cid]:>13}: {len(class_images[cid])}")

# ------------- 2. run inference on the selected images -------------
all_img_paths = []
for cid in range(len(CLASS_NAMES)):
    for img_path, _ in class_images[cid]:
        if img_path not in all_img_paths:
            all_img_paths.append(img_path)

print(f"\nRunning inference on {len(all_img_paths)} images...")
inference_results = model.predict(
    source=[str(p) for p in all_img_paths],
    imgsz=IMAGE_SIZE,
    device=0 if torch.cuda.device_count() >= 1 else "cpu",
    conf=cfg.conf,
    iou=cfg.iou,
    max_det=cfg.max_det,
    verbose=False,
)

annotated = {}
for r, ipath in zip(inference_results, all_img_paths):
    annotated[ipath.name] = r.plot(conf=True, labels=True)

# ------------- 3. display the 6x6 grid (12 classes x 3 samples = 36 images) -------------
N_ROWS, N_COLS = 6, 6
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(14, 14))
axes = axes.flatten()
idx = 0
for cid in range(len(CLASS_NAMES)):
    for img_path, _ in class_images[cid]:
        if idx >= N_ROWS * N_COLS:
            break
        img_bgr = annotated.get(img_path.name)
        if img_bgr is not None:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(img_rgb)
        axes[idx].set_title(CLASS_NAMES[cid], fontsize=7, fontweight="bold", color=CLASS_COLORS[cid])
        axes[idx].axis("off")
        idx += 1
for j in range(idx, len(axes)):
    axes[j].axis("off")
plt.tight_layout(pad=0.5)
plt.subplots_adjust(wspace=0.05, hspace=0.25)
plt.savefig(OUTPUT_DIR / "detection_grid_36.png", dpi=150)
plt.show()

# ------------- 4. correctness log -- per-image, per-class -------------
def compute_iou_log(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0:
        return 0.0
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return interArea / float(boxAArea + boxBArea - interArea)


print("\n" + "=" * 80)
print("  INFERENCE CORRECTNESS LOG -- Per-Image")
print("=" * 80)

class_correct = {i: 0 for i in range(len(CLASS_NAMES))}
class_missed = {i: 0 for i in range(len(CLASS_NAMES))}
class_total = {i: 0 for i in range(len(CLASS_NAMES))}

for cid in range(len(CLASS_NAMES)):
    for img_path, lbl_path in class_images[cid]:
        class_total[cid] += 1
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        # ground-truth class IDs and boxes for this image
        gt_classes_in_img = set()
        gt_boxes = []
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    gcls = int(float(parts[0]))
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = int((cx - bw / 2) * w); y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w); y2 = int((cy + bh / 2) * h)
                    gt_classes_in_img.add(gcls)
                    gt_boxes.append((gcls, x1, y1, x2, y2))

        # predicted class IDs and boxes for this image
        pred_classes_in_img = set()
        pred_boxes = []
        for r, ipath in zip(inference_results, all_img_paths):
            if ipath.name == img_path.name:
                if r.boxes is not None:
                    for box_data in r.boxes.data.cpu().numpy():
                        px1, py1, px2, py2, conf_v, pcls = box_data
                        pred_classes_in_img.add(int(pcls))
                        pred_boxes.append((int(pcls), int(px1), int(py1), int(px2), int(py2)))
                break

        # does any GT box of class cid match a pred of the SAME class at IoU >= 0.5?
        found = False
        for gcls, gx1, gy1, gx2, gy2 in gt_boxes:
            if gcls != cid:
                continue
            for pcls, px1, py1, px2, py2 in pred_boxes:
                if pcls != cid:
                    continue
                iou = compute_iou_log([gx1, gy1, gx2, gy2], [px1, py1, px2, py2])
                if iou >= 0.5:
                    found = True
                    break
            if found:
                break

        short_name = img_path.name[:35]
        if found:
            class_correct[cid] += 1
            print(f"  [OK]  {CLASS_NAMES[cid]:>13} | {short_name:<35} | GT:{gt_classes_in_img}  Pred:{pred_classes_in_img}")
        else:
            class_missed[cid] += 1
            print(f"  [MISS] {CLASS_NAMES[cid]:>13} | {short_name:<35} | GT:{gt_classes_in_img}  Pred:{pred_classes_in_img}")

# ---- summary table ----
print("\n" + "-" * 55)
print(f"  {'Class':<15} {'Correct':>7} {'Missed':>7} {'Accuracy':>9}")
print("  " + "-" * 44)
for cid in range(len(CLASS_NAMES)):
    total = class_total[cid]
    if total > 0:
        acc = class_correct[cid] / total * 100
        print(f"  {CLASS_NAMES[cid]:<15} {class_correct[cid]:>7} {class_missed[cid]:>7} {acc:>9.1f}%")
    else:
        print(f"  {CLASS_NAMES[cid]:<15} {0:>7} {0:>7} {'N/A':>9}")

total_correct = sum(class_correct.values())
total_images = sum(class_total.values())
print("  " + "-" * 44)
print(f"  {'OVERALL':<15} {total_correct:>7} {total_images - total_correct:>7} {total_correct / max(1, total_images) * 100:>9.1f}%")
print("-" * 55)


## 11. Interpretation (write-up notes for the report)

- The BYOL arm answers the Part B question at ρ = 0.2: does label-free pretraining on the 80% SSL
  pool refund part of the 80% annotation budget? Compare this notebook's test mAP50–95 with the
  random-init and COCO-init runs from the separate baselines notebook (same protocol, only the
  backbone initialisation differs).
- The **COCO baseline is the harder anchor** (it already embeds general object knowledge); the
  advantage of a domain SSL backbone usually shows at low label fractions — exactly the regime Part B
  studies.
- Failure clusters to verify from the grid and correctness log: small objects (caterpillars), crowded
  scenes (ants, bees), and the visually confusable Slugs/Snails pair (classes 8/9).
- All outputs are reproducible: config JSON, surgery report, run folder, `detection_performance.csv`,
  `test_per_class_metrics.csv`, and the 36-image panel.

## 12. Export — Detector for the Tracking Notebook

Copy the best **BYOL-initialised** detector to the working directory and write a run manifest so
Notebook 05 (tracking) can attach the exact checkpoint. To hand over, publish the whole output
directory as a Kaggle dataset.

In [ ]:
final_export = OUTPUT_DIR / "byol_init_yolo12m_20pct_detector.pt"
shutil.copy2(SSL_BEST, final_export)

manifest = {
    "method": "BYOL -> YOLOv12m @ rho=0.20",
    "seed": cfg.seed,
    "epochs": EPOCHS,
    "ssl_checkpoint": str(CKPT),
    "surgery": json.loads((OUTPUT_DIR / "weight_surgery_report.json").read_text(encoding="utf-8")),
    "best_detector": str(SSL_BEST),
    "performance_csv": str(OUTPUT_DIR / "detection_performance.csv"),
    "ref_parta_test_map50_95": cfg.parta_test_map50_95,
    "export": str(final_export),
}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Output directory:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUTPUT_DIR))
